# Библиотеки

In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import seaborn as sns

## 9.2 Генеральная совокупность и выборка

In [22]:
filepath = './data/1Sample_complex.xlsx'

s = pd.read_excel(filepath, sheet_name='Survey Data array')
p = pd.read_excel(filepath, sheet_name='School by classes', skipfooter=1)

In [23]:
s.head()

,ID родителей,Класс,NPS
0,185,8,6
1,112,7,5
2,1215,0,10
3,144,9,8
4,856,7,7


In [24]:
p.head()

,Классы,Количество учеников,Полнота семьи
0,11,16,2.00
1,10,21,2.00
2,9,33,1.97
3,8,37,1.86
4,7,35,1.86


In [25]:
p['Population'] = p['Количество учеников'] * p['Полнота семьи']

In [26]:
Pop = p['Population'].sum().round()

Pop

np.float64(1216.0)

In [27]:
prop = 0.5 # Доля успеха\Доля признака
error = 0.05 # Погрешность
zscore = 1.96 # Z-значение для 95% доверия

SSize = int( ((zscore**2 * prop*(1 - prop)) / (error**2) ) / (1 + (((zscore**2 * prop * (1 - prop)) / (error**2)) - 1) / Pop))
SSize

292

In [28]:
if SSize <= len(s):
    print(f"Ваша выборка достаточна по размеру. Надо {SSize}, а у Вас {len(s)}.")
else:
    print(f"Размер Вашей выборки {len(s)}, а надо {SSize}. Добавьте {SSize - len(s)} наблюдений.")

Размер Вашей выборки 96, а надо 292. Добавьте 196 наблюдений.


In [29]:
p['Perc'] = (p['Population']/Pop).round(2)

In [31]:
s1 = (s['Класс'].value_counts(normalize=True)).round(2).reset_index()

In [32]:
s1

,Класс,proportion
0,0,0.29
1,3,0.15
2,4,0.11
3,7,0.11
4,6,0.09
5,8,0.09
6,5,0.08
7,9,0.06


In [33]:
p = p.merge(s1, left_on='Классы', right_on='Класс', how='left') 

In [35]:
p['SvsP'] = p['Perc'] - p['proportion']

In [37]:
SE = (p['Perc'] * (1 - p['Perc']) / len(s)).pow(0.5)
p['Critical'] = 'yes'

p.loc[(p['proportion']>=(p['Perc']-zscore*SE)) & (p['proportion']<=(p['Perc']+zscore*SE)), 'Critical'] = 'no'

In [52]:
s2 = (s['Класс'].value_counts(normalize=False)).round(2).reset_index()

In [64]:
def sample_size(Pop):
    prop = 0.5
    error = 0.05
    zscore = 1.96
    return int( ((zscore**2 * prop*(1 - prop)) / (error**2) ) / (1 + (((zscore**2 * prop * (1 - prop)) / (error**2)) - 1) / Pop))

(s2
 .merge(p.loc[:, ['Класс', 'Population']],  how='left', left_on='Класс', right_on='Класс')
 .assign(MinSize = lambda df: df.Population.apply(sample_size)
        , Diff = lambda df: df.MinSize - df['count']
        , SizeStatus = lambda df: df['Diff'].apply(lambda x: 'yes' if x > 0 else 'no')
        )
)

,Класс,count,Population,MinSize,Diff,SizeStatus
0,0,28,260.00,155,127,yes
1,3,14,122.50,93,79,yes
2,4,11,110.40,85,74,yes
3,7,11,65.10,55,44,yes
4,6,9,79.20,65,56,yes
5,8,9,68.82,58,49,yes
6,5,8,50.00,44,36,yes
7,9,6,65.01,55,49,yes


# end